# 重回帰分析

重回帰は複数特徴量の寄与を同時に推定できる最小構成の予測モデルです。単回帰より現実的な設定で、解釈可能性と実装容易性を両立できます。


## 背景と目的

実データは複数要因が同時に効くため、単一説明変数だけで予測すると係数解釈が歪みます。

重回帰を理解すると、特徴量追加が予測と解釈に与える効果を定量的に評価できます。

行列表現・最小二乗解・多重共線性指標を順に確認します。


In [ ]:
import numpy as np
np.random.seed(11)

n = 120
x1 = np.random.randn(n)
x2 = 0.6 * x1 + 0.8 * np.random.randn(n)  # 相関あり
eps = 0.5 * np.random.randn(n)
y = 1.2 + 2.0 * x1 - 1.5 * x2 + eps

X = np.column_stack([np.ones(n), x1, x2])


In [ ]:
# 最小二乗解: beta = (X^T X)^(-1) X^T y
beta = np.linalg.pinv(X.T @ X) @ X.T @ y
pred = X @ beta
mse = np.mean((pred - y) ** 2)

print('beta_hat =', np.round(beta, 6))
print('train MSE=', round(mse, 6))


In [ ]:
# VIFで多重共線性を確認
# VIF_j = 1 / (1 - R_j^2)

def vif(x_target, x_other):
    Xo = np.column_stack([np.ones(len(x_other)), x_other])
    b = np.linalg.pinv(Xo.T @ Xo) @ Xo.T @ x_target
    pred = Xo @ b
    ss_res = np.sum((x_target - pred) ** 2)
    ss_tot = np.sum((x_target - x_target.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    return 1.0 / (1.0 - r2)

vif_x1 = vif(x1, x2)
vif_x2 = vif(x2, x1)
print('VIF(x1)=', round(vif_x1, 6), 'VIF(x2)=', round(vif_x2, 6))


係数推定値だけでなく VIF を併せて見ることで、予測精度と解釈安定性の両方を評価できます。
